# Output Validation

> **Source:** `repo1/security_patterns.py` → `demo_output_validation()`


## Imports


In [ ]:
import re
from typing import Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable
from dotenv import load_dotenv


## Setup


In [ ]:
load_dotenv()


## Class: `PIIDetector`


In [ ]:
class PIIDetector:
    """Detect and mask personally identifiable information."""

    PATTERNS = {
        "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",
        "phone": r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b",
        "ssn": r"\b\d{3}-\d{2}-\d{4}\b",
        "credit_card": r"\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b",
        "ip_address": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b",
    }

    def detect(self, text: str) -> dict[str, list[str]]:
        """Detect PII in text."""
        found = {}
        for pii_type, pattern in self.PATTERNS.items():
            matches = re.findall(pattern, text)
            if matches:
                found[pii_type] = matches
        return found

    def mask(self, text: str) -> str:
        """Mask PII in text."""
        masked = text
        for pii_type, pattern in self.PATTERNS.items():
            if pii_type == "email":
                masked = re.sub(pattern, "[EMAIL REDACTED]", masked)
            elif pii_type == "phone":
                masked = re.sub(pattern, "[PHONE REDACTED]", masked)
            elif pii_type == "ssn":
                masked = re.sub(pattern, "[SSN REDACTED]", masked)
            elif pii_type == "credit_card":
                masked = re.sub(pattern, "[CARD REDACTED]", masked)
            elif pii_type == "ip_address":
                masked = re.sub(pattern, "[IP REDACTED]", masked)
        return masked


## Class: `OutputValidator`


In [ ]:
class OutputValidator:
    """Validate LLM outputs before returning to user."""

    def __init__(self):
        self.pii_detector = PIIDetector()

    def validate(self, output: str) -> tuple[bool, str, Optional[str]]:
        """
        Validate output.
        Returns: (is_valid, cleaned_output, reason_if_invalid)
        """
        # Check for PII leakage
        pii_found = self.pii_detector.detect(output)
        if pii_found:
            cleaned = self.pii_detector.mask(output)
            return False, cleaned, f"PII detected and masked: {list(pii_found.keys())}"

        # Check for harmful content patterns
        harmful_patterns = [
            r"here('s| is) (how|the way) to (hack|steal|attack)",
            r"password is",
            r"api[_\s]?key",
        ]

        for pattern in harmful_patterns:
            if re.search(pattern, output, re.IGNORECASE):
                return (
                    False,
                    "[CONTENT BLOCKED]",
                    "Potentially harmful content detected",
                )

        return True, output, None


## Demo: Output Validation


In [ ]:
def demo_output_validation():
    """Demonstrate output validation."""

    validator = OutputValidator()

    outputs = [
        "The capital of France is Paris.",
        "Contact support at help@company.com for assistance.",
        "Here's how to hack into the system...",
    ]

    print("\nOutput Validation Demo:\n")

    for output in outputs:
        is_valid, cleaned, reason = validator.validate(output)
        status = "✅ VALID" if is_valid else "⚠️ CLEANED"
        print(f"{status}: {output[:50]}...")
        if reason:
            print(f"   Reason: {reason}")
            print(f"   Cleaned: {cleaned[:50]}...")


## Run


In [ ]:
demo_output_validation()
